# DAY4 과제 01 : 기초문제

데이터 파일 : `packaging_line.csv` (포장 라인 A · B · C 에서 배치 12개씩, 모두 36개 배치의 처리시간 기록)



- 유의수준은 모든 문제에서 α = 0.05 로 한다
- 결론은 두 줄로 쓴다 — 통계적 결론(귀무가설을 기각한다 / 기각하지 못한다)과 사실적 결론
- 기각하지 못하면 "다르다고 볼 근거가 부족하다"로 쓴다

| 컬럼 | 설명 |
|---|---|
| batch_id | 배치 번호 (A01 ~ C12) |
| line | 포장 라인 (A / B / C) |
| processing_time_sec | 배치 처리시간 (초) |

In [ ]:
%pip install -q koreanize-matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
from scipy import stats

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
data_folder = "/content/drive/MyDrive/HD-AI/data"
csv_path = os.path.join(data_folder, "packaging_line.csv")

df = pd.read_csv(csv_path)

data = df.copy()

## 1부. 일원분산분석 : 포장 라인 세 곳의 처리시간 비교

### 1. 라인별 배치 수 · 평균 · 표준편차를 한 표로 구하시오. (소수 둘째 자리까지)

In [ ]:
# 힌트 : groupby("line")["processing_time_sec"].agg(["count", "mean", "std"]).round(2)



### 2. 라인별 처리시간을 `A`, `B`, `C` 세 변수로 꺼내고, 라인마다 Shapiro-Wilk 정규성 검정의 p값을 구하시오.

- 귀무가설(H₀): 이 라인의 처리시간은 정규분포를 따른다.
- 대립가설(H₁): 이 라인의 처리시간은 정규분포를 따르지 않는다.

In [ ]:
# 힌트 : data.loc[data["line"] == "A", "processing_time_sec"], stats.shapiro(값).pvalue



### 3. Levene 검정으로 세 라인의 분산이 같은지 확인하시오.

- 귀무가설(H₀): 세 라인 처리시간의 모분산은 같다. (σA² = σB² = σC²)
- 대립가설(H₁): 적어도 한 라인의 모분산이 다르다.

In [ ]:
# 힌트 : stats.levene(A, B, C).pvalue



### 4. 일원분산분석으로 세 라인의 평균 처리시간이 같은지 검정하고, F 와 p값을 구하시오.

- 귀무가설(H₀): 세 라인의 모평균 처리시간은 같다. (μA = μB = μC)
- 대립가설(H₁): 적어도 한 라인의 모평균 처리시간이 다르다. (적어도 한 μ가 다르다)

In [ ]:
# 힌트 : stats.f_oneway(A, B, C) → .statistic, .pvalue



### 5. 4번에서 귀무가설을 기각했다면, Tukey HSD 사후검정으로 어느 라인끼리 다른지 확인하시오.

아래 두 방법 중 **하나를 골라** 쓴다.
- 방법 1 : `pairwise_tukeyhsd` (statsmodels)
- 방법 2 : `stats.tukey_hsd` (scipy)

쌍마다
- 귀무가설(H₀): 두 라인의 모평균 처리시간은 같다. (예 : μA = μC)
- 대립가설(H₁): 두 라인의 모평균 처리시간은 다르다. (예 : μA ≠ μC)

In [ ]:
# 힌트 : 방법 1 → from statsmodels.stats.multicomp import pairwise_tukeyhsd, pairwise_tukeyhsd(data["processing_time_sec"], data["line"]) / 방법 2 → stats.tukey_hsd(A, B, C), .pvalue[0, 2]



## 2부. 비율 검정 : 한 집단 · 두 집단

### 6. 출하 제품 120개를 검사했더니 불량이 13개였다. 기존 불량률 기준은 5%다. 표본 불량률, 정확 이항검정의 p값, 모비율의 95% 신뢰구간을 구하시오.

- 귀무가설(H₀): 출하 제품의 모집단 불량률은 5%다. (p = 0.05)
- 대립가설(H₁): 출하 제품의 모집단 불량률은 5%가 아니다. (p ≠ 0.05)

In [ ]:
# 힌트 : stats.binomtest(k=불량 수, n=전체 수, p=0.05), 결과.proportion_ci(confidence_level=0.95) → .low, .high



### 7. 주간조와 야간조에서 제품을 100개씩 검사했다. 주간조는 불량 17개, 야간조는 불량 9개였다.

(1) 아래 교차표를 만들고, 기대빈도가 모두 5 이상인지 확인하시오.

| 근무조 | 불량 | 정상 |
|---|---|---|
| 주간 | 17 | 83 |
| 야간 | 9 | 91 |

(2) 카이제곱 검정의 χ² 와 p값을 구하시오.

(3) 불량률 차이(주간 − 야간)와 그 95% 신뢰구간을 구하시오. (`method="wald"`)

- 귀무가설(H₀): 주간조와 야간조의 모집단 불량률은 같다. (p_주간 = p_야간)
- 대립가설(H₁): 주간조와 야간조의 모집단 불량률은 다르다. (p_주간 ≠ p_야간)

In [ ]:
# 힌트 : pd.DataFrame({"불량": [...], "정상": [...]}, index=["주간", "야간"]), stats.contingency.expected_freq(표), stats.chi2_contingency(표), from statsmodels.stats.proportion import confint_proportions_2indep



## 3부. 카이제곱 검정 : 적합도 · 동질성 · 독립성

### 8. 라벨 오류 100건을 유형별로 세었더니 정보 누락 42건, 위치 오류 38건, 인쇄 흐림 20건이었다. 과거 비율은 50% · 30% · 20%다. 기대빈도를 구하고 적합도 검정의 χ² 와 p값을 구하시오.

- 귀무가설(H₀): 라벨 오류 유형의 모비율은 과거 비율과 같다. (p₁ = 0.50, p₂ = 0.30, p₃ = 0.20)
- 대립가설(H₁): 적어도 한 오류 유형의 모비율이 과거 비율과 다르다.

In [ ]:
# 힌트 : 기대빈도 = 관측빈도 합계 × 과거 비율, stats.chisquare(f_obs=관측, f_exp=기대)



### 9. 교대조 1조 · 2조 · 3조에서 제품을 200개씩 정해서 뽑아 검사했다. 불량은 각각 10개, 12개, 24개였다. 교차표를 만들고 기대빈도를 확인한 뒤, 동질성 검정의 χ² 와 p값을 구하시오.

- 귀무가설(H₀): 세 교대조의 모집단 불량률은 같다. (p₁ = p₂ = p₃)
- 대립가설(H₁): 적어도 한 교대조의 모집단 불량률이 다르다.

In [ ]:
# 힌트 : 정상 = 200 − 불량, pd.DataFrame, stats.contingency.expected_freq(표), stats.chi2_contingency(표)



### 10. 한 달 동안 나온 불량 200건을 설비와 불량 유형 두 가지로 분류했다. 비율표(설비별 100%)를 구하고, 독립성 검정의 χ² 와 p값을 구하시오.

| 설비 | 스크래치 | 치수 | 이물 |
|---|---|---|---|
| 1호기 | 38 | 27 | 15 |
| 2호기 | 42 | 33 | 45 |

- 귀무가설(H₀): 설비와 불량 유형은 독립이다(관련이 없다). 설비가 달라도 불량 유형 비율은 같다. (P(설비 ∩ 유형) = P(설비) × P(유형))
- 대립가설(H₁): 설비와 불량 유형은 독립이 아니다(관련이 있다). 설비에 따라 불량 유형 비율이 다르다. (P(설비 ∩ 유형) ≠ P(설비) × P(유형))

In [ ]:
# 힌트 : 비율표 = 표.div(표.sum(axis=1), axis=0) * 100, stats.contingency.expected_freq(표), stats.chi2_contingency(표)



## 확인문제

앞의 결과를 보고 답을 1~2문장으로 적으시오.

### 확인 1. 2번 · 3번 결과로 볼 때, 4번에서 일원분산분석을 써도 되는 이유는 무엇인가?

**답 :**

### 확인 2. 4번의 통계적 결론과 사실적 결론을 쓰시오.

**답 :**

### 확인 3. 5번에서 차이가 있는 쌍은 어디이며, 어느 라인이 몇 초 더 긴가?

**답 :**

### 확인 4. 6번의 통계적 결론과 사실적 결론을 쓰시오.

**답 :**

### 확인 5. 7번에서 표본 불량률은 주간이 8%p 높았다. 그런데도 결론이 어떻게 나왔는지 신뢰구간과 함께 쓰시오.

**답 :**

### 확인 6. 8번의 통계적 결론과 사실적 결론을 쓰시오.

**답 :**

### 확인 7. 9번의 통계적 결론과 사실적 결론을 쓰시오.

**답 :**

### 확인 8. 10번의 통계적 결론과 사실적 결론을 쓰시오.

**답 :**

## 개념 확인 (객관식)

알맞은 답의 번호를 하나 고르시오.

### 개념 1. 일원분산분석에서 귀무가설을 기각했다. 이 결과로 말할 수 있는 것은?

① 모든 집단의 평균이 서로 다르다  
② 적어도 한 집단의 평균이 다르다  
③ A와 C의 평균이 다르다  
④ 모든 집단의 평균이 같다

**답 :**

### 개념 2. 일원분산분석에서 기각한 뒤, 어느 집단끼리 다른지 확인하는 사후검정은?

① Shapiro-Wilk 검정  
② Levene 검정  
③ Tukey HSD  
④ 적합도 검정

**답 :**

### 개념 3. 세 집단 모두 정규성은 크게 어긋나지 않지만, Levene 검정에서 분산이 다르다고 나왔다. 평균 비교에 알맞은 방법은?

① 일원분산분석 `stats.f_oneway(a, b, c)`  
② Welch ANOVA `stats.f_oneway(a, b, c, equal_var=False)`  
③ Fisher 정확검정  
④ 적합도 검정

**답 :**

### 개념 4. 정규성이 크게 어긋난 세 집단을 비교할 때 쓰는 검정은?

① Kruskal-Wallis 검정  
② 2×2 카이제곱 검정  
③ 정확 이항검정  
④ Levene 검정

**답 :**

### 개념 5. "제품 200개 중 불량 16개, 불량률이 기준 5%와 다른가"를 검정하는 함수는?

① `stats.binomtest`  
② `stats.f_oneway`  
③ `stats.levene`  
④ `stats.tukey_hsd`

**답 :**

### 개념 6. 이번 달 불량 유형별 건수가 과거 비율과 맞는지 확인하는 검정은?

① 독립성 검정  
② 동질성 검정  
③ 적합도 검정  
④ 일원분산분석

**답 :**

### 개념 7. 라인 A · B · C 에서 제품을 300개씩 정해서 뽑아, 세 라인의 불량률이 같은지 확인하는 검정은?

① 적합도 검정  
② 동질성 검정  
③ 정확 이항검정  
④ Kruskal-Wallis 검정

**답 :**

### 개념 8. 불량 200건 한 표본을 뽑아 '설비'와 '불량 유형' 두 가지로 분류한 뒤, 두 변수가 관련이 있는지 확인하는 검정은?

① 독립성 검정  
② 적합도 검정  
③ 일원분산분석  
④ Tukey HSD

**답 :**

### 개념 9. 카이제곱 검정 전에 확인했더니 기대빈도가 5 미만인 칸이 있었다. 알맞은 대처는?

① 그대로 카이제곱 검정을 한다  
② 건수표 대신 비율표를 넣는다  
③ 범주를 합치거나, 2×2 표라면 Fisher 정확검정을 쓴다  
④ 유의수준을 0.10 으로 바꾼다

**답 :**

### 개념 10. 검정 결과 p = 0.18 이었다 (α = 0.05). 알맞은 결론은?

① 귀무가설이 참임이 증명되었다  
② 귀무가설을 기각하지 못한다. 차이가 있다고 볼 근거가 부족하다  
③ 귀무가설을 기각한다  
④ 대립가설이 참이다

**답 :**